## Make various [element/element] vs. [Fe/H] (probably) plots 

In [1]:


from __future__ import print_function


import matplotlib
matplotlib.use('pdf')
savefig=True

import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
start = time.time()

import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import bensby_plotting as bp
import saga_plotting as sap

import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs




print(os.getcwd())

all_avg
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv
/Users/BenKaiser/Desktop/radial_velocity_calculations


In [2]:
figure_output_dir='/Users/BenKaiser/Desktop/'
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [3]:
lodders_abund_file='Lodders2020_solarsystem_abundances.csv'
solar_system_object_file='20220304_solar_system_body_abundances_select_names.csv'
#solar_system_object_file='20210713_solar_system_body_abundances_all_names.csv'
crust_file='20220304_continental_crust_vals_only.csv'
dp_arrow_file='20221110_WD_DP_abundances_arrowlengths.csv'




In [4]:
base_el='Ca'
#base_el='Na'
#base_el='H'
#base_el='O'
#base_el='Fe'

#el_list=['Li','C','Na','Mg','K','Ca','Cr','Fe']

el_list=['Li','Na','Mg','K','Ca','Cr','Fe'] #all WD-detected elements by atomic number
#el_list=['Li','Na','Mg','K','Cr','Fe']
ssp=True

#xabund='[Fe/H]'
xabund='[Ca/H]'

In [5]:
if ssp:
    #wd_abund_file='20220304_WD_SSP_abundances.csv'
    wd_abund_file='20221109_WD_SSP_abundances_J1636recovered.csv'
    show_dp=True
else:
    #wd_abund_file='20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'
    wd_abund_file='20221109_all_wd_abundances_7030MCages_DR3kinematics_missingabundsJ1636added.csv'

    show_dp=False




In [6]:
print(os.getcwd())

print(wd_abund_file)
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
lodders_table=Table.read(lodders_abund_file)
bodies_table=Table.read(solar_system_object_file)
crust_table=Table.read(crust_file)
dp_arrow_table=Table.read(dp_arrow_file)

lodders_table.add_index('element')
wd_abund_table.add_index('name')
bodies_table.add_index('name')
dp_arrow_table.add_index('name')



#wd_num_abund_table=Table.read(wd_num_abund_file)
#wd_num_abund_table.add_index('name')

limit_length=0.3 #length of limit error bars on plots
limit_indicator=99. #value above which if the absolute value of the error on a measurement is above it indicates it should be a limit




/Users/BenKaiser/Desktop/radial_velocity_calculations
20221109_WD_SSP_abundances_J1636recovered.csv


In [7]:
use_indices=np.where(bodies_table['show']==1)
use_bodies_table=bodies_table[use_indices]
#use_wd_indices=np.where((wd_abund_table['show']==1)and (wd_abund_table['show_geo']==1))
use_wd_indices=np.where(wd_abund_table['show_li_evo']==1)
use_wd_abund_table=wd_abund_table[use_wd_indices]
use_wd_indices=np.where(use_wd_abund_table['show']==1)
use_wd_abund_table=use_wd_abund_table[use_wd_indices]

In [8]:

t_step=5

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
met_color='#1ca1f2'
met_size=3
#ci_size=6
ci_size=4
wd_size=10
dp_alpha=0.5
ci_leg_size=9
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
#arrow_width=0.03
#arrow_width=0.07

background_alpha=0.4

arrow_line=4
figure_text_size=6
default_offset=[0.05,0.00]
annot_line_weight=0.03


object_spacing=0.2
arrow_alpha=0.4
arrow_head_length_mult=1.5 #pyplot default arrow head length multiple of the head_width
arrow_width=object_spacing*0.9
arrow_head_width=arrow_width
xmin=-4.5
xmax=0.75
ymin=-1.75
ymax=3.25
first_obj_pos=-4




show_all_ssobj_names=False

In [9]:
def plot_wd_errorbar(x_coord, el1el2, el1el2_err,name, selected_marker=wd_marker, markersize=wd_size, label='', color='b'):
    if label=='':
        #label=name #commented this on 2021-11-12 to try to keep star points out of legend
        pass 
    else:
        pass
    uplims=False
    lolims=False
    if np.abs(el1el2_err)> limit_indicator:
        if el1el2_err > 0:
            lolims=True
        elif el1el2_err < 0:
            uplims=True
        else:
            print("This shouldn't print el1el2_err")
        el1el2_err= limit_length
    else:
        #no limit indicators are present for the 2 relative abundances input
        pass
    plt.errorbar(x_coord, el1el2, yerr= el1el2_err, uplims=uplims, lolims=lolims,  color=color,marker=selected_marker,  markersize=markersize,linestyle='None',zorder=2)
    plt.errorbar(x_coord,el1el2, label=label, marker=selected_marker, markersize=markersize, color=color,linestyle='None')
    return

In [10]:
def plot_CI_chondrite():
    plt.errorbar(0, 0,  color=met_color,marker=met_marker,  markersize=ci_size,linestyle='None',zorder=2)
    return

In [12]:

for i,name in enumerate(el_list):
    if name==base_el:
        continue
    else:
        pass
    spt.initiate_science_plot()
    #spt.start_ApJ_fig(width_cols=1,constrained_layout=True,width_height=[1,0.75])
    spt.start_ApJ_fig(width_cols=1,constrained_layout=True,width_height=[1,0.75*0.8])

    #plt.figure(figsize=(10.,7.25))
    ax=plt.subplot()
    print(i,name)
    num_wds=len(use_wd_abund_table)
    el_string=name.lower()+'/'+base_el.lower()
    original_el_string=el_string
    #plot_CI_chondrite(start_pos+object_spacing,name,label=CI_label)
    
    
    #bp.plot_el1el2_FeH(name,base_el,error_bars=False,alpha=background_alpha,color='r')
    #sap.plot_el_abunds('[Fe/H]','['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='b',require_uncertainties=True)
    try:
        #sap.plot_el_abunds('[Ca/H]','['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='grey',require_uncertainties=True)
        #sap.plot_el_abunds('[Fe/H]','['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='grey',require_uncertainties=True)
        sap.plot_el_abunds(xabund,'['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='grey',require_uncertainties=True,base_layer=True)


    except KeyError as error:
        print("KeyError made it all the way out:",error)
    #if ((name=='Li') and (base_el=='Ca')):
    #    #and the x-axis is [Ca/H]
    #    xvals=np.linspace(-4,1,100)
    #    LiCa_CI=lodders_table.loc["Li"]['A_el']-lodders_table.loc[base_el]['A_el']
    #    ACa_vals=xvals+lodders_table.loc[base_el]['A_el']
    #    yvals=2.72-ACa_vals-LiCa_CI
    #    plt.plot(xvals,yvals,linestyle=':',color='k')
    if ((name=='Li') and (xabund.split('/')[0].replace('[','')==base_el)):
        #and the x-axis is [Ca/H]
        #xvals=np.linspace(-4,1,100)
        xvals=np.linspace(xmin,xmax,100)
        LiCa_CI=lodders_table.loc["Li"]['A_el']-lodders_table.loc[base_el]['A_el']
        ACa_vals=xvals+lodders_table.loc[base_el]['A_el']
        yvals=2.72-ACa_vals-LiCa_CI
        plt.plot(xvals,yvals,linestyle=':',color='k')
    else:
        pass
        
    #sap.plot_el_abunds('[Na/H]','['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='b',require_uncertainties=True)

    try:
        el1_CI=lodders_table.loc[name]['A_el']
        el2_CI=lodders_table.loc[base_el]['A_el']
        el1_CI_err=lodders_table.loc[name]['A_el_err']
        el2_CI_err=lodders_table.loc[base_el]['A_el_err']
        el1el2_CI=el1_CI-el2_CI
        el1el2_CI_err=np.sqrt(el1_CI_err**2+el2_CI_err**2)
        plot_CI_chondrite()
    except KeyError as error:
        print("KeyError:",error)
    if ssp:
        el_string='ssp_'+el_string
        marker=ssp_marker
        markersize=ci_size
    else:
        marker=wd_marker
        markersize=wd_size
    for j,row in enumerate(use_wd_abund_table):
        object_pos=first_obj_pos+(j*object_spacing)
        print(j,row['name'])
        try:
            el1el2=row[el_string]
            norm_el1el2=el1el2-el1el2_CI
            plt.axhline(y=norm_el1el2,label=row['display_name'],color=row['plot_color'],linestyle='--',zorder=1)
            plot_wd_errorbar(object_pos, norm_el1el2,row[el_string+'_err'],row['display_name'],color=row['plot_color'],selected_marker=marker,markersize=markersize)
            dp_row=dp_arrow_table.loc[row['name']]
            #print('dp_row',dp_row)
            if ((row['show_dp']) and (show_dp) and (original_el_string!='k/ca')):
                #excludes the K/Ca decreasing phase arrows because those are too short to get out from behind the symbols.
                if (np.abs(dp_row['dp_'+original_el_string]) < arrow_head_length_mult*arrow_head_width):
                    print(el_string,'arrow_head_length would have been less than the arrow length...',dp_row['dp_'+original_el_string] ,"<",arrow_head_length_mult*arrow_head_width,row['show_dp'])
                    arrow_head_length=dp_row['dp_'+original_el_string]
                else:
                    arrow_head_length=arrow_head_length_mult*arrow_head_width
                plt.arrow(object_pos,row[el_string]-el1el2_CI,0,dp_row['dp_'+original_el_string],color=row['plot_color'],width=arrow_width, alpha=arrow_alpha,length_includes_head=True,linewidth=0,head_width=arrow_head_width,zorder=0.9 )
                #print(el_string,row['name'],object_pos,row[el_string],object_pos,dp_row['dp_'+original_el_string])
            else:
                print('show_dp is false',row['show_dp'], 'so not adding arrow')
        
        except KeyError:
            pass
        #plt.text(-2.5,norm_el1el2+0.01,row['display_name'],fontsize=figure_text_size)
    #plt.ylim(-3.25,2.5)
    #if name!="Li":
        #plt.ylim(-1.1,2.5)
    #plt.xlim(-2.75,0.75)
    
    #plt.ylim(-3.25,3)
    #plt.xlim(-4.5,0.75)
    #plt.xlim(xmin,0.75)
    plt.xlim(xmin,xmax)
    #plt.ylim(ymin,ymax)
    if savefig:
        print(os.getcwd())
        os.chdir(figure_output_dir)
        print(os.getcwd())
        start = time.time()
        print(start)
        time_string=str(start).split('.')[0]
        if ssp:
            plt.savefig(name+base_el+"_element_evo_ssp"+'_vs_'+ xabund.replace('[','').replace('/','').replace(']','') +'_'+time_string+'.pdf')#plt.grid(True)
        else:
            plt.savefig(name+base_el+"_element_evo"+'_vs_'+ xabund.replace('[','').replace('/','').replace(']','') +'_'+time_string+'.pdf')#plt.grid(True)
        print("Figure saved")
    else:
        pass
    plt.show()
    #plt.text(edge_spot+0.5*el_space,0,name)
#plt.ylabel('log(Z/Ca)')
#plt.xlim()
#plt.ylim(-3.5,2)
#x_ticks=np.arange(0.5*el_space,len(el_list)*el_space,el_space)
#print(x_ticks)
#ax.set_xticks(x_ticks)
#ax.set_xticklabels(el_list)
#plt.legend(loc='best',fontsize=7)
#if savefig:
    #print(os.getcwd())
    #os.chdir(figure_output_dir)
    #print(os.getcwd())
    #start = time.time()
    #print(start)
    #time_string=str(start).split('.')[0]
    #if ssp:
        #plt.savefig("element_evo_ssp"+'_vs_FeH'+'_'+time_string+'.pdf')#plt.grid(True)
    #else:
        #plt.savefig("all_elements"+'_vs_'+base_el+'_'+time_string+'.pdf')#plt.grid(True)
    #print("Figure saved")
#else:
    #pass
plt.show()

0 Li
MS_only: True
--
KeyError: '[Li/Ca]'
el1 [Li hopefully is Li
0 WDJ1644-0449
1 SDSSJ1330+6435
2 WDJ1824+1213
3 WDJ2317+1830
show_dp is false 0 so not adding arrow
4 LHS2534
5 WDJ2356-209
6 SDSSJ1636+1619
/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1668475364.7103202
Figure saved
1 Na
MS_only: True


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:115: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


--
KeyError: '[Na/Ca]'
0 WDJ1644-0449
1 SDSSJ1330+6435
2 WDJ1824+1213
3 WDJ2317+1830
show_dp is false 0 so not adding arrow
4 LHS2534
5 WDJ2356-209
6 SDSSJ1636+1619
/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1668475365.745263
Figure saved
2 Mg
MS_only: True


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:115: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


--
KeyError: '[Mg/Ca]'
0 WDJ1644-0449
1 SDSSJ1330+6435
2 WDJ1824+1213
3 WDJ2317+1830
show_dp is false 0 so not adding arrow
4 LHS2534
5 WDJ2356-209
6 SDSSJ1636+1619
/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1668475367.1317399
Figure saved
3

...

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)


Figure saved
6 Fe
MS_only: True


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:115: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


--
KeyError: '[Fe/Ca]'
0 WDJ1644-0449
1 SDSSJ1330+6435
2 WDJ1824+1213
3 WDJ2317+1830
show_dp is false 0 so not adding arrow
4 LHS2534
5 WDJ2356-209
6 SDSSJ1636+1619
/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1668475370.3717802
Figure saved


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:115: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:139: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


### Plotting of the [el/H] vs. Teff and log g correlations


for i,name in enumerate(el_list):
    if name==base_el:
        continue
    else:
        pass
    spt.initiate_science_plot()
    spt.start_ApJ_fig(width_cols=1,constrained_layout=True,width_height=[0.5,0.5])
    #plt.figure(figsize=(10.,7.25))
    ax=plt.subplot()
    print(i,name)
    num_wds=len(use_wd_abund_table)
    el_string=name.lower()+'/'+base_el.lower()
    #plot_CI_chondrite(start_pos+object_spacing,name,label=CI_label)
    
    
    #bp.plot_el1el2_FeH(name,base_el,error_bars=False,alpha=background_alpha,color='r')
    #sap.plot_el_abunds('[Fe/H]','['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='b',require_uncertainties=True)
    try:
        sap.plot_el_vs_param('Teff','['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='b',require_uncertainties=True)
        #sap.plot_el_abunds('[Fe/H]','['+name+'/'+base_el+']',errorbars=True,alpha=background_alpha,MS_only=True,color='b',require_uncertainties=True)

    except KeyError as error:
        print("KeyError made it all the way out:",error)
    if ((name=='Li') and (base_el=='Ca')):
        #and the x-axis is [Ca/H]
        xvals=np.linspace(-4,1,100)
        LiCa_CI=lodders_table.loc["Li"]['A_el']-lodders_table.loc[base_el]['A_el']
        ACa_vals=xvals+lodders_table.loc[base_el]['A_el']
        yvals=2.72-ACa_vals-LiCa_CI
        plt.plot(xvals,yvals,linestyle=':',color='k')
    else:
        pass
        
    #sap.plot_el_abunds('[Na/H]','['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='b',require_uncertainties=True)

    try:
        el1_CI=lodders_table.loc[name]['A_el']
        el2_CI=lodders_table.loc[base_el]['A_el']
        el1_CI_err=lodders_table.loc[name]['A_el_err']
        el2_CI_err=lodders_table.loc[base_el]['A_el_err']
        el1el2_CI=el1_CI-el2_CI
        el1el2_CI_err=np.sqrt(el1_CI_err**2+el2_CI_err**2)
        #plot_CI_chondrite()
    except KeyError as error:
        print("KeyError:",error)
    if ssp:
        el_string='ssp_'+el_string
        marker=ssp_marker
        markersize=ci_size
    else:
        marker=wd_marker
        markersize=wd_size

        #plt.text(-2.5,norm_el1el2+0.01,row['display_name'],fontsize=figure_text_size)
    #plt.ylim(-3.25,2.5)
    #if name!="Li":
        #plt.ylim(-1.1,2.5)
    #plt.xlim(-2.75,0.75)
    
    #plt.ylim(-3.25,3)
    #plt.xlim(-4.5,0.75)

    if savefig:
        print(os.getcwd())
        os.chdir(figure_output_dir)
        print(os.getcwd())
        start = time.time()
        print(start)
        time_string=str(start).split('.')[0]
        if ssp:
            plt.savefig(name+base_el+"_element_evo_ssp"+'_vs_Teff'+'_'+time_string+'.pdf')#plt.grid(True)
        else:
            plt.savefig(name+base_el+"_element_evo"+'_vs_Teff'+'_'+time_string+'.pdf')#plt.grid(True)
        print("Figure saved")
    else:
        pass
    plt.show()
    #plt.text(edge_spot+0.5*el_space,0,name)
#plt.ylabel('log(Z/Ca)')
#plt.xlim()
#plt.ylim(-3.5,2)
#x_ticks=np.arange(0.5*el_space,len(el_list)*el_space,el_space)
#print(x_ticks)
#ax.set_xticks(x_ticks)
#ax.set_xticklabels(el_list)
#plt.legend(loc='best',fontsize=7)
#if savefig:
    #print(os.getcwd())
    #os.chdir(figure_output_dir)
    #print(os.getcwd())
    #start = time.time()
    #print(start)
    #time_string=str(start).split('.')[0]
    #if ssp:
        #plt.savefig("element_evo_ssp"+'_vs_FeH'+'_'+time_string+'.pdf')#plt.grid(True)
    #else:
        #plt.savefig("all_elements"+'_vs_'+base_el+'_'+time_string+'.pdf')#plt.grid(True)
    #print("Figure saved")
#else:
    #pass
plt.show()

### Gonna try doing element-to-element evolution plots
converted the cell below to be markdown so it doesn't execute in the future. Hypothetically it could have been good, but unfortunately log(Li/Ca) is super uncommon to be plottable with the requirements (and SAGA's relative lack of lithium measurements) so these plots were not that good.


for i,name in enumerate(el_list):
    if name==base_el:
        continue
    else:
        pass
    spt.initiate_science_plot()
    spt.start_ApJ_fig(width_cols=1,constrained_layout=True,width_height=[0.5,0.5])
    #plt.figure(figsize=(10.,7.25))
    ax=plt.subplot()
    print(i,name)
    num_wds=len(use_wd_abund_table)
    el_string=name.lower()+'/'+base_el.lower()
    #plot_CI_chondrite(start_pos+object_spacing,name,label=CI_label)
    
    
    #bp.plot_el1el2_FeH(name,base_el,error_bars=False,alpha=background_alpha,color='r')
    #sap.plot_el_abunds('[Fe/H]','['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='b',require_uncertainties=True)
    sap.plot_el_abunds('[Li/Ca]','['+name+'/'+base_el+']',errorbars=True,alpha=background_alpha,MS_only=True,color='b',require_uncertainties=True)
    #sap.plot_el_abunds('[Na/H]','['+name+'/'+base_el+']',errorbars=False,alpha=background_alpha,MS_only=True,color='b',require_uncertainties=True)


    el1_CI=lodders_table.loc[name]['A_el']
    el2_CI=lodders_table.loc[base_el]['A_el']
    el1_CI_err=lodders_table.loc[name]['A_el_err']
    el2_CI_err=lodders_table.loc[base_el]['A_el_err']
    el1el2_CI=el1_CI-el2_CI
    el1el2_CI_err=np.sqrt(el1_CI_err**2+el2_CI_err**2)
    
    loddersLiCa=lodders_table.loc["Li"]['A_el']-lodders_table.loc["Ca"]['A_el']
    
    
    
    plot_CI_chondrite()
    if ssp:
        el_string='ssp_'+el_string
        marker=ssp_marker
        markersize=ci_size
    else:
        marker=wd_marker
        markersize=wd_size
    for j,row in enumerate(use_wd_abund_table):
        object_pos=-2.7+(j*0.1)
        try:
            el1el2=row[el_string]
            norm_el1el2=el1el2-el1el2_CI
            #plt.axhline(y=norm_el1el2,label=row['display_name'],color=row['plot_color'],linestyle='--')
            plot_wd_errorbar(row['ssp_li/ca']-loddersLiCa, norm_el1el2,row[el_string+'_err'],row['display_name'],color=row['plot_color'],selected_marker=marker,markersize=markersize)
        except KeyError:
            pass
        #plt.text(-2.5,norm_el1el2+0.01,row['display_name'],fontsize=figure_text_size)
    #plt.ylim(-3.25,2.5)
    #if name!="Li":
        #plt.ylim(-1.1,2.5)
    #plt.xlim(-2.75,0.75)
    
    #plt.ylim(-3.25,3)
    #plt.xlim(-4.5,0.75)

    if savefig:
        print(os.getcwd())
        os.chdir(figure_output_dir)
        print(os.getcwd())
        start = time.time()
        print(start)
        time_string=str(start).split('.')[0]
        if ssp:
            plt.savefig(name+base_el+"_element_evo_ssp"+'_vs_LiCa'+'_'+time_string+'.pdf')#plt.grid(True)
        else:
            plt.savefig(name+base_el+"_element_evo"+'_vs_LiCa'+'_'+time_string+'.pdf')#plt.grid(True)
        print("Figure saved")
    else:
        pass
    plt.show()
    #plt.text(edge_spot+0.5*el_space,0,name)
#plt.ylabel('log(Z/Ca)')
#plt.xlim()
#plt.ylim(-3.5,2)
#x_ticks=np.arange(0.5*el_space,len(el_list)*el_space,el_space)
#print(x_ticks)
#ax.set_xticks(x_ticks)
#ax.set_xticklabels(el_list)
#plt.legend(loc='best',fontsize=7)
#if savefig:
    #print(os.getcwd())
    #os.chdir(figure_output_dir)
    #print(os.getcwd())
    #start = time.time()
    #print(start)
    #time_string=str(start).split('.')[0]
    #if ssp:
        #plt.savefig("element_evo_ssp"+'_vs_FeH'+'_'+time_string+'.pdf')#plt.grid(True)
    #else:
        #plt.savefig("all_elements"+'_vs_'+base_el+'_'+time_string+'.pdf')#plt.grid(True)
    #print("Figure saved")
#else:
    #pass
plt.show()